# Eval Base Qwen3 Models on Dependency Analysis

Evaluates **Qwen3-1.7B**, **Qwen3-4B**, and **Qwen3-8B** (Instruct) against the Gemini 3 Flash teacher.

**Pipeline:**
1. Load Gemini reference from `dep_analysis_labeled.json`
2. Clone 5 eval repos, sample N files per repo
3. Run each Qwen3 model via **vLLM** (`localhost:8000`)
4. Compute F1 / recall / precision vs Gemini

**Prerequisites:**
- `pip install -r requirements-eval.txt`
- vLLM serving a Qwen3 model:
  ```bash
  vllm serve Qwen/Qwen3-1.7B --max-model-len 16384
  ```
- Switch model between runs by restarting vLLM with a different model

In [1]:
%pip install -q -r requirements-eval.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, sys, json, time, random, subprocess
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Project root = this notebook's directory
ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

from lib.collector import collect_files
from lib.llm_client import llm_extract
from lib.languages import get_language_config

print(f"Root: {ROOT}")

Root: /workspace/qwen3_distill


## Config

In [2]:
# ── vLLM endpoint ──
VLLM_URL = "http://localhost:8000/v1"

# ── Models to evaluate ──
# Run this notebook once per model. Before each run, start vLLM with that model.
# Set CURRENT_MODEL to match what vLLM is serving.
MODELS = {
    "qwen3-1.7b": {
        "hf_id": "Qwen/Qwen3-1.7B",
        "workers": 4,
        "max_tokens": 1024,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    "qwen3-4b": {
        "hf_id": "Qwen/Qwen3-4B",
        "workers": 4,
        "max_tokens": 1024,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    "qwen3-8b": {
        "hf_id": "Qwen/Qwen3-8B",
        "workers": 2,
        "max_tokens": 1024,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

# >>>> SET THIS to match the model currently loaded in vLLM <<<<
CURRENT_MODEL = "qwen3-8b"

N_SAMPLES = 50
MAX_CONTENT_LENGTH = 12000

print(f"Will evaluate: {CURRENT_MODEL} ({MODELS[CURRENT_MODEL]['hf_id']})")

Will evaluate: qwen3-8b (Qwen/Qwen3-8B)


## Paths & Repos

In [3]:
REPOS_FILE = ROOT / "data" / "eval_repos.json"
REPOS_DIR = ROOT / "repos"
RESULTS_DIR = ROOT / "results"
LABELED_DATA_PATH = ROOT / "data" / "dep_analysis_labeled.json"

REPOS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

with open(REPOS_FILE) as f:
    REPOS = json.load(f)

print("Eval repos:")
for r in REPOS:
    print(f"  {r['name']:15s} ({r['language']:12s}) {r['url']}")

Eval repos:
  garnet          (csharp      ) https://github.com/microsoft/garnet
  anime           (javascript  ) https://github.com/juliangarnier/anime
  llama.cpp       (cpp         ) https://github.com/ggml-org/llama.cpp
  langchain4j     (java        ) https://github.com/langchain4j/langchain4j
  checkmate       (python      ) https://github.com/quantifiedcode/checkmate


## Load Gemini reference from labeled data

In [4]:
def load_labeled_data(path):
    """Parse dep_analysis_labeled.json -> {rel_path: result_dict}."""
    with open(path) as f:
        data = json.load(f)

    results = {}
    for item in data:
        if not isinstance(item, dict) or "conversations" not in item:
            continue
        convs = item["conversations"]
        if len(convs) < 2:
            continue
        rel_path = None
        for line in convs[0]["value"].split("\n"):
            if line.strip().startswith("File:"):
                rel_path = line.strip()[5:].strip()
                break
        if rel_path is None:
            continue
        try:
            result = json.loads(convs[1]["value"])
        except (json.JSONDecodeError, TypeError):
            continue
        if isinstance(result, dict):
            results[rel_path] = result

    print(f"Loaded {len(results)} labeled entries")
    return results

labeled = load_labeled_data(LABELED_DATA_PATH)

Loaded 4640 labeled entries


## Clone repos & sample files

In [5]:
def clone_repo(url, dest):
    if dest.is_dir():
        print(f"  Already cloned: {dest.name}")
        return True
    print(f"  Cloning {url} ...")
    r = subprocess.run(["git", "clone", "--depth=1", url, str(dest)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ERROR: {r.stderr.strip()}")
        return False
    return True


def sample_files(root_dir, config, n, seed=42):
    all_files = collect_files(
        str(root_dir),
        extensions=config["extensions"],
        skip_dirs=config["skip_dirs"],
        skip_patterns=config.get("skip_patterns"),
    )
    random.seed(seed)
    return random.sample(all_files, min(n, len(all_files)))


for repo in REPOS:
    clone_repo(repo["url"], REPOS_DIR / repo["name"])

  Already cloned: garnet
  Already cloned: anime
  Already cloned: llama.cpp
  Already cloned: langchain4j
  Already cloned: checkmate


## Populate Gemini cache from labeled data

In [6]:
for repo in REPOS:
    name = repo["name"]
    config = get_language_config(repo["language"])
    repo_dir = REPOS_DIR / name
    results_dir = RESULTS_DIR / name
    results_dir.mkdir(exist_ok=True)

    files = sample_files(repo_dir, config, N_SAMPLES)
    gemini_path = results_dir / "gemini_results.json"

    existing = {}
    if gemini_path.exists():
        with open(gemini_path) as f:
            existing = json.load(f)

    added = 0
    for fp, _ in files:
        if fp not in existing and fp in labeled:
            existing[fp] = labeled[fp]
            added += 1

    with open(gemini_path, "w") as f:
        json.dump(existing, f, indent=2)

    covered = sum(1 for fp, _ in files if fp in existing)
    print(f"{name:15s}: {covered}/{len(files)} files have Gemini reference (+{added} new)")

garnet         : 50/50 files have Gemini reference (+0 new)
anime          : 50/50 files have Gemini reference (+0 new)
llama.cpp      : 49/50 files have Gemini reference (+0 new)
langchain4j    : 50/50 files have Gemini reference (+0 new)
checkmate      : 50/50 files have Gemini reference (+0 new)


## Run student model via vLLM

In [7]:
def run_model_on_files(files, prompt_template, default_result, model_cfg, cache_path):
    """Run vLLM inference on files with caching."""
    results = {}
    if cache_path.exists():
        with open(cache_path) as f:
            results = json.load(f)

    remaining = [(fp, c) for fp, c in files if fp not in results]
    if not remaining:
        print(f"    All {len(results)} files cached")
        return results

    print(f"    {len(remaining)} files to process (workers={model_cfg['workers']})")

    def process(args):
        fp, content = args
        prompt = prompt_template.replace(
            "FILEPATH_PLACEHOLDER", fp
        ).replace(
            "CONTENT_PLACEHOLDER", content[:MAX_CONTENT_LENGTH]
        )
        t0 = time.time()
        parsed = llm_extract(
            prompt,
            model=model_cfg["hf_id"],
            base_url=VLLM_URL,
            api_key="",
            max_tokens=model_cfg.get("max_tokens"),
            extra_body=model_cfg.get("extra_body"),
        )
        elapsed = time.time() - t0
        if parsed is None or not isinstance(parsed, dict):
            return fp, dict(default_result), False, elapsed
        result = {k: parsed.get(k, v) for k, v in default_result.items()}
        return fp, result, True, elapsed

    done = 0
    total = len(remaining)
    with ThreadPoolExecutor(max_workers=model_cfg["workers"]) as executor:
        futures = {executor.submit(process, item): item[0] for item in remaining}
        for future in as_completed(futures):
            fp, result, ok, elapsed = future.result()
            results[fp] = result
            done += 1
            status = 'OK' if ok else 'FAIL'
            print(f"      [{done}/{total}] {status} {fp} ({elapsed:.1f}s)")
            if done % 10 == 0:
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)

    with open(cache_path, "w") as f:
        json.dump(results, f, indent=2)
    return results

In [8]:
cfg = MODELS[CURRENT_MODEL]
print(f"Running {CURRENT_MODEL} ({cfg['hf_id']}) on all repos...\n")

all_student_results = {}
for repo in REPOS:
    name = repo["name"]
    config = get_language_config(repo["language"])
    repo_dir = REPOS_DIR / name
    results_dir = RESULTS_DIR / name

    files = sample_files(repo_dir, config, N_SAMPLES)
    cache_path = results_dir / f"qwen_{CURRENT_MODEL}_results.json"

    print(f"\n--- {name} ({repo['language']}) ---")
    all_student_results[name] = run_model_on_files(
        files, config["prompt"], config["default_result"], cfg, cache_path
    )

print("\nDone!")

Running qwen3-8b (Qwen/Qwen3-8B) on all repos...


--- garnet (csharp) ---
    50 files to process (workers=2)
      [1/50] OK benchmark/Resp.benchmark/ClientTypes.cs (1.0s)
      [2/50] OK libs/common/Networking/INetworkHandler.cs (1.2s)
      [3/50] OK libs/server/Storage/Functions/MainStore/VectorSessionFunctions.cs (2.5s)
  JSON parse error. Raw: {
  "namespace": "Tsavorite.benchmark",
  "definitions": ["IKeySetter", "TestLoader"],
  "usings": ["System", "System.Collections.Generic", "System.Diagnostics", "System.IO", "System.Linq", "System.Ru...
      [4/50] FAIL libs/storage/Tsavorite/cs/benchmark/YCSB.benchmark/TestLoader.cs (19.9s)
      [5/50] OK libs/server/AOF/AofHeader.cs (0.9s)
      [6/50] OK libs/common/SimpleObjectPool.cs (1.5s)
  JSON parse error. Raw: {
  "namespace": "Garnet.server",
  "definitions": ["HLL_DTYPE", "HyperLogLog"],
  "usings": ["System", "System.Collections.Generic", "System.Diagnostics", "System.Numerics", "System.Runtime.CompilerS...
      [7/50] FAI

## Compute metrics vs Gemini

In [9]:
def _norm(item):
    if isinstance(item, dict):
        return json.dumps(item, sort_keys=True)
    return str(item).strip().lower()


def compute_metrics(gemini_results, qwen_results, default_result):
    common = sorted(set(gemini_results) & set(qwen_results))
    fields = list(default_result.keys())
    field_metrics = {}

    for field in fields:
        g_counts, q_counts, recalls, precisions, exact = [], [], [], [], 0
        for fp in common:
            g_val = gemini_results[fp].get(field, [])
            q_val = qwen_results[fp].get(field, [])
            if isinstance(g_val, list):
                g_set = {_norm(x) for x in g_val}
                q_set = {_norm(x) for x in q_val}
            else:
                g_set = {str(g_val)} if g_val else set()
                q_set = {str(q_val)} if q_val else set()

            g_counts.append(len(g_set))
            q_counts.append(len(q_set))
            recalls.append(len(g_set & q_set) / len(g_set) if g_set else (1.0 if not q_set else 0.0))
            precisions.append(len(g_set & q_set) / len(q_set) if q_set else (1.0 if not g_set else 0.0))
            if g_set == q_set:
                exact += 1

        n = len(common)
        r = sum(recalls) / n if n else 0
        p = sum(precisions) / n if n else 0
        field_metrics[field] = {
            "gemini_avg": round(sum(g_counts) / n, 2) if n else 0,
            "qwen_avg": round(sum(q_counts) / n, 2) if n else 0,
            "recall": round(r, 4),
            "precision": round(p, 4),
            "f1": round(2 * r * p / (r + p), 4) if (r + p) else 0,
            "exact_match": round(exact / n, 4) if n else 0,
        }

    return {"files_compared": len(common), "fields": field_metrics}

In [10]:
print(f"=== Results for {CURRENT_MODEL} ({MODELS[CURRENT_MODEL]['hf_id']}) ===")
print()

all_metrics = {}
for repo in REPOS:
    name = repo["name"]
    config = get_language_config(repo["language"])
    results_dir = RESULTS_DIR / name

    gemini_path = results_dir / "gemini_results.json"
    student_path = results_dir / f"qwen_{CURRENT_MODEL}_results.json"

    if not gemini_path.exists() or not student_path.exists():
        print(f"{name}: SKIPPED (missing cache)")
        continue

    with open(gemini_path) as f:
        gemini_res = json.load(f)
    with open(student_path) as f:
        student_res = json.load(f)

    metrics = compute_metrics(gemini_res, student_res, config["default_result"])
    all_metrics[name] = metrics

    print(f"--- {name} ({metrics['files_compared']} files) ---")
    print(f"  {'Field':<18} {'G.avg':>6} {'Q.avg':>6}  {'Recall':>7} {'Prec':>7} {'F1':>6} {'Exact':>6}")
    for field, m in metrics["fields"].items():
        print(f"  {field:<18} {m['gemini_avg']:>6.1f} {m['qwen_avg']:>6.1f}"
              f"  {m['recall']:>7.1%} {m['precision']:>7.1%} {m['f1']:>6.3f} {m['exact_match']:>6.1%}")
    print()

# Overall avg
if all_metrics:
    print("=== OVERALL (averaged across repos) ===")
    fields = list(next(iter(all_metrics.values()))["fields"].keys())
    for field in fields:
        f1s = [m["fields"][field]["f1"] for m in all_metrics.values() if field in m["fields"]]
        recalls = [m["fields"][field]["recall"] for m in all_metrics.values() if field in m["fields"]]
        print(f"  {field:<18} avg F1={sum(f1s)/len(f1s):.3f}  avg Recall={sum(recalls)/len(recalls):.1%}")

=== Results for qwen3-8b (Qwen/Qwen3-8B) ===

--- garnet (50 files) ---
  Field               G.avg  Q.avg   Recall    Prec     F1  Exact
  namespace             1.0    0.7    74.0%   74.0%  0.740  74.0%
  definitions           1.5    1.2    70.9%   68.2%  0.695  62.0%
  usings                1.3    2.4    47.0%   30.4%  0.369  18.0%
  used_classes          6.7    6.7    48.7%   38.9%  0.433  20.0%
  global_usings         0.0    0.0   100.0%  100.0%  1.000 100.0%

--- anime (50 files) ---
  Field               G.avg  Q.avg   Recall    Prec     F1  Exact
  definitions           1.5    4.8    53.0%   49.0%  0.509  44.0%
  references            7.3    7.8    97.3%   95.4%  0.964  84.0%

--- llama.cpp (49 files) ---
  Field               G.avg  Q.avg   Recall    Prec     F1  Exact
  local_includes        2.6    2.6    85.7%   85.0%  0.853  83.7%
  system_includes       1.9    1.9    86.0%   87.4%  0.866  79.6%
  forward_declarations    0.4    0.9    81.2%   81.4%  0.813  79.6%

--- langcha

## Save report

In [11]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = RESULTS_DIR / f"comparison_{CURRENT_MODEL}_{ts}"

with open(str(report_path) + ".json", "w") as f:
    json.dump(all_metrics, f, indent=2)

with open(str(report_path) + ".txt", "w") as f:
    f.write(f"Dependency Analysis - Model Comparison Report\n")
    f.write(f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Teacher   : Gemini 3 Flash (from labeled data)\n")
    f.write(f"Student   : {MODELS[CURRENT_MODEL]['hf_id']} (vLLM)\n\n")
    for repo_name, metrics in all_metrics.items():
        f.write(f"\n{'=' * 60}\nREPO: {repo_name} ({metrics['files_compared']} files)\n{'=' * 60}\n")
        f.write(f"  {'Field':<18} {'G.avg':>6} {'Q.avg':>6}  {'Recall':>7} {'Prec':>7} {'F1':>6} {'Exact':>6}\n")
        for field, m in metrics["fields"].items():
            f.write(f"  {field:<18} {m['gemini_avg']:>6.1f} {m['qwen_avg']:>6.1f}"
                    f"  {m['recall']:>7.1%} {m['precision']:>7.1%} {m['f1']:>6.3f} {m['exact_match']:>6.1%}\n")

print(f"Saved: {report_path}.json")
print(f"Saved: {report_path}.txt")

Saved: /workspace/qwen3_distill/results/comparison_qwen3-8b_20260313_050121.json
Saved: /workspace/qwen3_distill/results/comparison_qwen3-8b_20260313_050121.txt
